# Guía de Elementos y Conceptos — Práctica 2: Histogramas, Filtrado y Detección de Bordes

Esta guía sintetiza los **elementos técnicos, funciones, interpretaciones teóricas y plantillas de código** trabajados en la Práctica 2 (y su tutorial previo), organizados por temas para consulta rápida en el examen.

### Índice de Bloques
1. **[Histogramas de Intensidad (RGB y Escala de Grises) y Efecto de `nbins`](#1-histogramas)**
2. **[Efecto de la Escala Espacial (`rescale`) sobre la Imagen y el Histograma](#2-escala-histograma)**
3. **[Convolución Lineal (`scipy.ndimage.convolve`): 1D, 2D y Canales RGB (`conv_coor`)](#3-convolucion-lineal)**
4. **[Filtros de Suavizado Ponderados (Gaussiano) vs No Lineales (Mediana)](#4-gaussiano-vs-mediana)**
5. **[Detección de Bordes: Prewitt, Sobel y Canny (Integrados y Manuales)](#5-deteccion-bordes)**
6. **[Frecuencias Espaciales, Imágenes Híbridas y Anonimización de Regiones (ROI)](#6-hibridas-y-roi)**

In [1]:
# Imports estándar utilizados en toda la Práctica 2
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from matplotlib import pyplot as plt
from skimage import io, exposure, filters, morphology, img_as_float, img_as_ubyte
from skimage.color import rgb2gray
from skimage.transform import resize, rescale
from scipy.ndimage import convolve
from skimage.filters import gaussian, median, sobel, prewitt
from skimage.morphology import disk, rectangle, diamond
from skimage.feature import canny

<a id='1-histogramas'></a>
---
## 1. Histogramas de Intensidad (RGB y Escala de Grises) y Efecto de `nbins`

Un **histograma** muestra la distribución de intensidades de una imagen agrupando los valores de los píxeles en `nbins` intervalos (eje X = intensidad, eje Y = frecuencia o número de píxeles).

### 1.1 Dos Métodos para Calcular y Mostrar Histogramas
| Método | Función de cálculo | Función de dibujo | Detalle clave a recordar |
| :--- | :--- | :--- | :--- |
| **1. Matplotlib** | `ax.hist(canal.ravel(), bins=nbins)` | Integrada en `hist()` | Es obligatorio aplanar la matriz 2D a un vector 1D con **`.ravel()`** |
| **2. Scikit-image** | `hist, bins = exposure.histogram(canal, nbins=nbins)` | `ax.bar(bins, hist, width=bins[1]-bins[0])` | Conviene convertir antes la imagen a **`float`** (`img.astype(float)/255` o `img_as_float`) para que `nbins` genere exactamente ese número de barras |

### 1.2 Interpretación Teórica de los Histogramas
- **Efecto de `nbins` (de `2` a `256`):**
  - **Pocos bins (ej. `nbins=8`):** Agrupa rangos muy anchos de intensidad en cada barra; resume la tendencia general pero oculta picos locales y detalles.
  - **Muchos bins (ej. `nbins=128` o `256`):** Muestra la distribución exacta nivel por nivel (mayor resolución fotométrica, menor conteo por barra).
  - **Equilibrio óptimo:** Valores intermedios como **`32` o `64`** permiten distinguir claramente los modos principales (objetos frente a fondo) sin ruido excesivo.
- **Relación entre los canales R, G, B y la escala de grises:**
  - Cada canal R, G y B muestra la distribución de su componente de color. Como `rgb2gray` calcula una media ponderada ($Y = 0.2125 R + 0.7154 G + 0.0721 B$), el histograma en escala de grises presenta una distribución promediada y más suave que combina los tres canales.

In [2]:
# Plantilla de función rgb_im_hist con ambos métodos listos para usar
def rgb_im_hist(img, nbins=8, metodo='skimage'):
    if not 2 <= nbins <= 256:
        raise ValueError("nbins must be between 2 and 256.")

    img_gray = rgb2gray(img)
    fig, ax = plt.subplots(1, 5, figsize=(20, 5))
    ax[0].imshow(img); ax[0].axis("off"); ax[0].set_title("Original Image")

    canales = [img[:, :, 0], img[:, :, 1], img[:, :, 2], img_gray]
    titulos = ["Red Channel", "Green Channel", "Blue Channel", "Grayscale"]

    for i, (ch, titulo) in enumerate(zip(canales, titulos), start=1):
        if metodo == 'plt':
            ax[i].hist(ch.ravel(), bins=nbins)
        else:
            hist, bins = exposure.histogram(ch, nbins=nbins)
            ax[i].bar(bins, hist, width=bins[1] - bins[0])
        ax[i].set_title(titulo)
        ax[i].set_xlabel("Intensity")
        ax[i].set_ylabel("Frequency")

    plt.tight_layout()
    plt.show()

<a id='2-escala-histograma'></a>
---
## 2. Efecto de la Escala Espacial (`rescale`) sobre la Imagen y el Histograma

Al modificar la **resolución espacial** de una imagen con `skimage.transform.rescale(img, factor)`:

1. **Reducción de tamaño (*Downsampling*, ej. `rescale(img, 1/20)`):**
   - Se fusionan y promedian bloques grandes de píxeles vecinos, provocando una **fuerte pérdida de detalle**.
   - El número total de píxeles cae drásticamente ($20^2 = 400$ veces menos píxeles), por lo que las alturas de las barras (frecuencias) son mucho menores.
   - Al promediarse los valores, desaparecen niveles extremos y quedan menos valores únicos: **el histograma se vuelve disperso (*sparse*, con huecos)** y su rango dinámico se estrecha.
2. **Ampliación de tamaño (*Upsampling*, ej. `rescale(img, 20)`):**
   - Se interpolan nuevos píxeles a partir de los vecinos existentes.
   - **La forma del histograma se mantiene idéntica a la original**, pero las cuentas de frecuencia en el eje Y son mucho mayores ($400$ veces más píxeles). La interpolación aumenta la cantidad de píxeles pero no crea nuevos niveles extremos de intensidad.

In [3]:
# Comparación del histograma de una imagen a escala original, reducida (1/20) y ampliada (20)
def comparar_escalas_histograma(img_gray, factor=20, nbins=64):
    img_small = rescale(img_gray, 1 / factor)
    img_big   = rescale(img_gray, factor)

    plt.figure(figsize=(15, 4))
    for idx, (im, label) in enumerate([(img_gray, "Original"), (img_small, f"{factor}x Smaller"), (img_big, f"{factor}x Larger")], start=1):
        hist, bins = exposure.histogram(im, nbins=nbins)
        plt.subplot(1, 3, idx)
        # Multiplicamos bins por 255 si la imagen está en float [0, 1] para visualizar el eje X en [0, 255]
        plt.bar(255 * bins, hist, width=2)
        plt.title(label); plt.xlabel("Pixel Intensity"); plt.ylabel("Frequency"); plt.xlim([0, 255])
    plt.tight_layout()
    plt.show()

<a id='3-convolucion-lineal'></a>
---
## 3. Convolución Lineal (`scipy.ndimage.convolve`): 1D, 2D y Canales RGB

### 3.1 Principios de la Convolución de Suavizado (Filtros de Media)
- **Normalización obligatoria de la máscara:** Antes de aplicar `convolve(img, mask)` con un filtro de suavizado, hay que dividir la máscara por la suma de todos sus elementos:
  $$\text{mask} = \frac{\text{mask}}{\sum \text{mask}}$$
  - Si **no se normaliza**, cada píxel resultante será la suma (no el promedio) de sus vecinos, multiplicando los valores por $\sum \text{mask}$ y saturando el brillo.
  - Es imprescindible que la máscara y la imagen resultante sean de tipo **`float`** para no truncar los pesos decimales a `0`.

### 3.2 Efecto Direccional de las Máscaras 1D vs 2D
- **Máscara 1D Horizontal (`shape = (1, N)`):** Debe definirse como matriz 2D `[[1, 1, ..., 1]]`. Suaviza solo horizontalmente (difumina bordes verticales y conserva bordes horizontales). Reduce moderadamente el rango `[min, max]`.
- **Máscara 1D Vertical (`shape = (N, 1)`):** Se obtiene transponiendo la horizontal (`mask_h1d.T`). Suaviza solo verticalmente (difumina bordes horizontales y conserva verticales).
- **Máscara 2D Uniforme (`shape = (N, N)`, `np.ones((N, N))`):** Suaviza en ambas direcciones a la vez sobre $N \times N$ píxeles, produciendo un desenfoque global mucho mayor y una mayor reducción del rango de intensidades.

### 3.3 Efecto de la Convolución sobre el Histograma y según la Escala
- **Sobre el histograma:** Al promediar píxeles vecinos, el filtro elimina las altas frecuencias y los valores extremos, **reduciendo la varianza** de intensidades y concentrando el histograma en un pico más estrecho en la zona media.
- **Según la escala de la imagen (con máscara fija `7×7`):**
  - En la **imagen reducida (`1/20`)**, la máscara `7×7` abarca casi toda la imagen, destruyendo los detalles y comprimiendo el histograma en unos pocos bins centrales.
  - En la **imagen ampliada (`20×`)**, la máscara `7×7` es diminuta respecto al tamaño de los objetos, por lo que apenas desenfoca y el histograma mantiene casi intacta la distribución original.

In [4]:
# 1. Creación y aplicación de máscaras normalizadas 1D Horizontal, 1D Vertical y 2D
def aplicar_convoluciones_basicas(img_gray, N=15):
    # 1D Horizontal (1 x N)
    mask_h1d = np.ones((1, N), dtype=float)
    mask_h1d = mask_h1d / np.sum(mask_h1d)
    out_h1d = convolve(img_gray, mask_h1d)

    # 1D Vertical (N x 1) -> Transpuesta
    mask_v1d = mask_h1d.T
    out_v1d = convolve(img_gray, mask_v1d)

    # 2D Uniforme (N x N)
    mask_2d = np.ones((N, N), dtype=float)
    mask_2d = mask_2d / np.sum(mask_2d)
    out_2d = convolve(img_gray, mask_2d)
    return out_h1d, out_v1d, out_2d

# 2. Convolución 2D aplicada independientemente a cada canal de una imagen RGB
def conv_coor(image, mask):
    result_image = np.zeros_like(image, dtype=float)
    mask = mask / np.sum(mask)  # Normalizar para que sume 1
    for i in range(3):
        result_image[:, :, i] = convolve(image[:, :, i], mask)
    return result_image

<a id='4-gaussiano-vs-mediana'></a>
---
## 4. Filtros de Suavizado Ponderados (Gaussiano) vs No Lineales (Mediana)

| Filtro | Tipo | Función y Parámetros | Ventajas y Comportamiento |
| :--- | :--- | :--- | :--- |
| **Media (Uniforme)** | Lineal (pesos iguales) | `convolve(img, mask/mask.sum())` | Desenfoca bordes y convierte un píxel ruidoso aislado en una mancha alrededor |
| **Gaussiano** | Lineal ponderado | `gaussian(img, sigma=...)` | Da más peso al píxel central según una campana de Gauss controlada por $\sigma$. Suavizado más natural e isótropo |
| **Mediana** | **No lineal** (estadístico) | `median(img, footprint)` | Sustituye el píxel por la mediana del vecindario. **Elimina por completo el ruido sal y pimienta** y preserva bordes nítidos |

### 4.1 Elección de $\sigma$ en el Filtro Gaussiano
- **$\sigma = 0.5$:** Suavizado muy leve; conserva todos los detalles finos (ej. cuernos del órix y textura del pelaje).
- **$\sigma = 1.0$:** **Equilibrio óptimo** para reducir el ruido de fondo manteniendo bien definidas las estructuras principales y los cuernos.
- **$\sigma = 2.0$ y $\sigma = 3.0$:** Desenfoque excesivo; borra estructuras delgadas (los cuernos se funden con el cielo) y deja solo siluetas grandes.

### 4.2 Efecto de la Forma y Tamaño (`footprint`) en el Filtro de Mediana
Los elementos estructurantes de `skimage.morphology` definen la vecindad sobre la que se calcula la mediana:
- **`morphology.disk(r)`:** Círculo de **radio `r`** píxeles. Con `disk(5)`, elimina bien el ruido y conserva el cuerpo del órix, pero **borra detalles finos y delgados como los cuernos** (porque ocupan menos de la mitad de los píxeles del disco).
- **`morphology.rectangle(alto, ancho)`:** Rectángulo de dimensiones `alto × ancho`. Con `rectangle(5, 20)`, al ser muy grande y alargado, **elimina los cuernos e introduce distorsión horizontal**.
- **`morphology.diamond(r)`:** Rombo de **radio `r`** píxeles. Con `diamond(3)`, al ser más compacto, logra el **mejor equilibrio entre los filtros de mediana Probados**, reduciendo el ruido sin hacer desaparecer los cuernos.

In [5]:
# Aplicación de filtros Gaussianos y de Mediana con distintos elementos estructurantes
def aplicar_gaussiano_y_mediana(img_gray):
    img_fl = img_as_float(img_gray)

    # Filtros Gaussianos
    g_05 = gaussian(img_fl, sigma=0.5)
    g_1  = gaussian(img_fl, sigma=1.0)
    g_2  = gaussian(img_fl, sigma=2.0)
    g_3  = gaussian(img_fl, sigma=3.0)

    # Filtros de Mediana (disk y diamond reciben radio; rectangle recibe alto, ancho)
    m_disk = median(img_fl, morphology.disk(5))
    m_rect = median(img_fl, morphology.rectangle(5, 20))
    m_diam = median(img_fl, morphology.diamond(3))
    return (g_05, g_1, g_2, g_3), (m_disk, m_rect, m_diam)

<a id='5-deteccion-bordes'></a>
---
## 5. Detección de Bordes: Prewitt, Sobel y Canny (Integrados y Manuales)

Un borde se produce donde hay un cambio brusco de intensidad (gradiente alto). Antes de aplicar cualquier detector de bordes, la imagen debe estar en **escala de grises 2D** (`rgb2gray(img[:, :, :3])`).

### 5.1 Comparativa de Detectores de Bordes
| Detector | Función en `skimage` | Tipo de Salida | Características y Limitaciones |
| :--- | :--- | :--- | :--- |
| **Prewitt** | `filters.prewitt(gray)` | **Escala de grises (`float`)** (magnitud del gradiente) | Promedio uniforme + derivada. Produce **bordes gruesos**, sin umbralizar, y es **muy sensible al ruido y texturas** |
| **Sobel** | `filters.sobel(gray)` | **Escala de grises (`float`)** (magnitud del gradiente) | Da peso doble (`2`) a la fila/columna central respecto a Prewitt. Bordes gruesos y sensible al ruido (requiere umbral posterior) |
| **Canny** | `feature.canny(gray, sigma=1)` | **Binaria (`bool`)** (bordes de 1 píxel) | Incluye: 1) Suavizado Gaussiano ($\sigma$), 2) Gradiente, 3) **Supresión de no-máximos** (adelgaza bordes) y 4) **Umbralización por histéresis**. Bordes limpios y finos, aunque sensible a la elección de $\sigma$ |

### 5.2 Dos Preguntas Clave de Examen sobre Bordes
1. **¿Se deben normalizar las máscaras de Prewitt o Sobel dividiendo por su suma?**
   - **No.** Son filtros derivativos que miden diferencias de intensidad, no promedios. La suma de los coeficientes de sus máscaras es **`0`** (para dar respuesta `0` en zonas de color uniforme).
2. **¿Mejora la detección aplicar un suavizado Gaussiano previo?**
   - **Sí.** Suavizar antes de detectar bordes elimina el ruido de alta frecuencia y las texturas finas (como barba, poros de la piel o césped) que generan falsos contornos. Sin embargo, un suavizado excesivo difumina también los bordes reales y hace perder detalles débiles.

### 5.3 Kernels Manuales de Prewitt y Sobel y Magnitud del Gradiente
- **Filtro horizontal $G_x$ (detecta bordes verticales):**
  $$\text{Prewitt } G_x = \begin{bmatrix} 1 & 0 & -1 \\ 1 & 0 & -1 \\ 1 & 0 & -1 \end{bmatrix}, \qquad \text{Sobel } G_x = \begin{bmatrix} 1 & 0 & -1 \\ 2 & 0 & -2 \\ 1 & 0 & -1 \end{bmatrix}$$
- **Filtro vertical $G_y$ (detecta bordes horizontales):** es la transpuesta $G_y = G_x^T$.
- **Magnitud del gradiente:** $|\nabla I| = \sqrt{G_x^2 + G_y^2}$.

In [6]:
# Implementación manual de Prewitt y Sobel mediante convolución
sobel_filter_x   = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]])
sobel_filter_y   = np.array([[1, 2, 1],  [0, 0, 0],  [-1, -2, -1]])

prewitt_filter_x = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]])
prewitt_filter_y = np.array([[1, 1, 1],  [0, 0, 0],  [-1, -1, -1]])

def detect_edges(img, filter_x, filter_y, filters_normalized=False):
    float_img = img_as_float(img)
    # Derivada en X (bordes verticales) y derivada en Y (bordes horizontales)
    vertical_edges_img   = convolve(float_img, filter_x)
    horizontal_edges_img = convolve(float_img, filter_y)

    # Magnitud del gradiente
    edges_img = np.sqrt(vertical_edges_img**2 + horizontal_edges_img**2)

    # Normalizar el resultado final al rango [0.0, 1.0] mediante min-max
    if not filters_normalized:
        min_val, max_val = np.min(edges_img), np.max(edges_img)
        edges_img = (edges_img - min_val) / (max_val - min_val)

    return img_as_ubyte(edges_img)

<a id='6-hibridas-y-roi'></a>
---
## 6. Frecuencias Espaciales, Imágenes Híbridas y Anonimización de Regiones (ROI)

### 6.1 Separación en Bajas y Altas Frecuencias e Imágenes Híbridas
Toda imagen $I$ se puede descomponer en la suma de sus bajas frecuencias (formas globales suaves) y sus altas frecuencias (bordes y texturas finas):
1. **Filtro Paso Bajo $L(I)$:** Se obtiene suavizando con un filtro Gaussiano: `low = gaussian(img, sigma=4)`.
2. **Filtro Paso Alto $H(I)$:** Se obtiene restando las bajas frecuencias a la imagen original:
   $$H(I) = I - L(I)$$
3. **Imagen Híbrida $\text{Hybrid}(I_1, I_2)$:**
   - Primero hay que asegurarse de que $I_1$ e $I_2$ tengan **exactamente las mismas dimensiones** usando `resize(img, (300, 300))`.
   - Se suman las bajas frecuencias de una con las altas frecuencias de la otra:
     $$\text{Hybrid}(I_1, I_2) = 2 \cdot \left(0.5 \cdot L(I_1) + 0.5 \cdot H(I_2)\right)$$
   - De cerca se percibe $H(I_2)$ (bordes finos) y de lejos se percibe $L(I_1)$ (estructura global).

### 6.2 Anonimización de Información Sensible (Difuminado Local sobre una ROI)
Para difuminar únicamente una región rectangular (ej. matrículas o caras) sin alterar el resto de la imagen:
1. Convertir la imagen completa a `float` con `img_float = img_as_float(image)`.
2. Localizar las coordenadas de la esquina superior izquierda `(r1, c1)` e inferior derecha `(r2, c2)`.
3. Extraer la sub-región con slicing: `region = img_float[r1:r2, c1:c2]`.
4. Aplicar convolución RGB con una máscara uniforme grande (ej. `np.ones((25, 25))`) mediante `conv_coor(region, mask)`.
5. Sustituir la región filtrada en la imagen original: `img_float[r1:r2, c1:c2] = blurred_region`.

In [7]:
# 1. Plantilla para generar imágenes híbridas a partir de dos imágenes RGB
def crear_imagenes_hibridas(img1, img2, size=(300, 300), sigma=4):
    i1 = resize(img1, size)
    i2 = resize(img2, size)

    # Paso bajo (Gaussiano) y Paso alto (Original - Paso bajo)
    i1_low = gaussian(i1, sigma=sigma)
    i1_high = i1 - i1_low

    i2_low = gaussian(i2, sigma=sigma)
    i2_high = i2 - i2_low

    # Combinación ponderada al 50% compensada por factor 2
    hybrid_1 = 2 * (0.5 * i1_low + 0.5 * i2_high)
    hybrid_2 = 2 * (0.5 * i2_low + 0.5 * i1_high)
    return hybrid_1, hybrid_2

# 2. Plantilla para anonimizar una región rectangular (top_left = (fila_ini, col_ini), bottom_right = (fila_fin, col_fin))
def anonymize_region(image_float, top_left, bottom_right, mask_size=25):
    region = image_float[top_left[0]:bottom_right[0], top_left[1]:bottom_right[1]]
    mask = np.ones((mask_size, mask_size))
    blurred_region = conv_coor(region, mask)
    image_float[top_left[0]:bottom_right[0], top_left[1]:bottom_right[1]] = blurred_region